# VetAI — Train the Image Disease Model (Google Colab)

This notebook trains the photo-based disease detector (MobileNetV2) on your
Roboflow **Cattle Disease Detection** dataset, using a free GPU.

**Before you start:**
1. In Colab's menu: **Runtime → Change runtime type → GPU → Save**.
2. Upload your dataset **zip** to your Google Drive (e.g. to *MyDrive*).
3. Then run each cell below in order (click the ▶ button, top to bottom).

At the end it downloads `vetai_image_model.zip`. Unzip it and put the two files
inside your project at `backend/app/ml/`. That's it — the photo tab will work.

## Step 1 — Connect your Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Unzip your dataset
Edit `ZIP_PATH` to point to your uploaded zip, then run.

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/cattle_disease.zip'   # <-- CHANGE to your zip's path

import zipfile, os
os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/dataset')

# Auto-find the folder that actually holds train/valid/test
DATASET_DIR = None
for root, dirs, files in os.walk('/content/dataset'):
    if '_annotations.coco.json' in files or ({'train','valid','test'} & set(dirs)):
        DATASET_DIR = root
        break
print('Dataset folder:', DATASET_DIR)
assert DATASET_DIR, 'Could not find the dataset. Check ZIP_PATH.'

## Step 3 — Training code
Just run this cell (it defines how the data is read and the model is built).

In [ ]:
import json, os
from collections import Counter
import numpy as np
import tensorflow as tf

IMG_SIZE = 224
BATCH = 32
EPOCHS = 15
DISEASE_CLASSES = ['healthy', 'fmd', 'lsd']
AUTOTUNE = tf.data.AUTOTUNE

def map_category_to_disease(name):
    n = name.lower()
    if 'normal' in n or 'healthy' in n: return 'healthy'
    if 'lumpy' in n: return 'lsd'
    if 'foot' in n or 'mouth' in n: return 'fmd'
    return None

def find_splits(d):
    s = {}
    for split in ['train','valid','test']:
        p = os.path.join(d, split)
        if os.path.isdir(p) and os.path.exists(os.path.join(p, '_annotations.coco.json')):
            s[split] = p
    if not s and os.path.exists(os.path.join(d,'_annotations.coco.json')):
        s['train'] = d
    return s

def load_split(split_dir):
    coco = json.load(open(os.path.join(split_dir,'_annotations.coco.json')))
    cat = {c['id']: c['name'] for c in coco['categories']}
    files = {im['id']: im['file_name'] for im in coco['images']}
    per_img = {}
    for a in coco['annotations']:
        per_img.setdefault(a['image_id'], []).append(a['category_id'])
    fps, labels, raw = [], [], Counter()
    for iid, fn in files.items():
        cats = per_img.get(iid, [])
        if not cats: continue
        raw_name = cat[Counter(cats).most_common(1)[0][0]]
        raw[raw_name] += 1
        dis = map_category_to_disease(raw_name)
        if dis is None: continue
        path = os.path.join(split_dir, fn)
        if os.path.exists(path):
            fps.append(path); labels.append(DISEASE_CLASSES.index(dis))
    return fps, labels, raw

def make_ds(fps, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((fps, labels))
    if training: ds = ds.shuffle(len(fps))
    def dec(p, y):
        img = tf.image.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return img, y
    ds = ds.map(dec, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x,y: (tf.image.random_flip_left_right(x), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x,y: (tf.keras.applications.mobilenet_v2.preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

print('Ready.')

## Step 4 — Train the model

In [ ]:
splits = find_splits(DATASET_DIR)
print('Splits found:', list(splits))
for s, p in splits.items():
    _, lab, raw = load_split(p)
    print(f'  {s}: {len(lab)} usable images ->', dict(Counter(DISEASE_CLASSES[i] for i in lab)))
    print('     raw classes:', dict(raw))

tr_fp, tr_lab, _ = load_split(splits['train'])
if 'valid' in splits:
    va_fp, va_lab, _ = load_split(splits['valid'])
else:
    cut = int(len(tr_fp)*0.85)
    tr_fp, va_fp = tr_fp[:cut], tr_fp[cut:]
    tr_lab, va_lab = tr_lab[:cut], tr_lab[cut:]

counts = np.bincount(tr_lab, minlength=len(DISEASE_CLASSES)); total = counts.sum()
class_weight = {i: (total/(len(DISEASE_CLASSES)*c) if c else 0.0) for i,c in enumerate(counts)}

base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, weights='imagenet')
base.trainable = False
inp = tf.keras.Input((IMG_SIZE,IMG_SIZE,3))
x = base(inp, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(len(DISEASE_CLASSES), activation='softmax')(x)
model = tf.keras.Model(inp, out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy')
model.fit(make_ds(tr_fp,tr_lab,True), validation_data=make_ds(va_fp,va_lab), epochs=EPOCHS, class_weight=class_weight, callbacks=[early])

## Step 5 — Check accuracy on the test set

In [ ]:
metrics = {}
if 'test' in splits:
    te_fp, te_lab, _ = load_split(splits['test'])
    if te_fp:
        loss, acc = model.evaluate(make_ds(te_fp,te_lab), verbose=0)
        metrics['test_accuracy'] = float(acc)
        print(f'Test accuracy: {acc:.3f}')
else:
    print('No separate test split; validation accuracy above is your guide.')

## Step 6 — Download the trained model
This creates `vetai_image_model.zip`. Unzip it and place **both** files into
your project folder `backend/app/ml/`.

In [ ]:
import shutil
os.makedirs('/content/out', exist_ok=True)
model.save('/content/out/image_model.keras')
json.dump({'classes': DISEASE_CLASSES, 'metrics': metrics}, open('/content/out/image_classes.json','w'), indent=2)
shutil.make_archive('/content/vetai_image_model', 'zip', '/content/out')
from google.colab import files
files.download('/content/vetai_image_model.zip')
print('Done — put image_model.keras and image_classes.json into backend/app/ml/')